# 04 · Training: two-stage LightGBM + decision tuning

**Folds** — 5 folds grouped by S1 id (deterministic hash), so all candidates of one S1 share a fold.

**Stage 1** — LightGBM binary classifier on the pair features. Fold models are fit on a sample of S1 entities (`train_frac`), then **every** train pair gets an out-of-fold probability `p1`. Scoring every pair (not just a sample) matters: target-competition in stage 2 needs the p1 of *all* S1s that compete for a record.

**Stage 2** — LightGBM on stage-1 features + `p1` + set context: the S1's p1 max/second/sum/rank, the best p1 any *other* S1 gives the target, and similarity of the candidate to the S1's top-1 / top-2 candidates (true matches are noisy copies of each other). Out-of-fold `p` for every pair.

**Decision** — grid on OOF scores with the exact macro F0.5: plain threshold vs. exclusivity + threshold vs. exclusivity + per-S1 expected-F0.5 subset selection. The winner is frozen in `artifacts/models/decision.json`.

Compute notes: the rows used to *fit* each stage come from the resource profile (`EF_PROFILE`: 16gb / 32gb / 64gb, or `EF_STAGE1_MAX_ROWS` / `EF_STAGE2_MAX_ROWS`). OOF scoring always covers every pair, streamed part by part.

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
# Every stage call below is checkpointed: re-running a notebook skips finished work.
# Production runs don't need the notebooks: `bash scripts/run_notebooks.sh` (see README).
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Optional overrides (or export EF_* variables before starting Jupyter):
# os.environ["EF_DEV_MODE"] = "1"      # small consistent slice (artifacts_dev/)
# os.environ["EF_PROFILE"] = "32gb"    # 16gb | 32gb | 64gb (default: auto from RAM)
# os.environ["EF_N_THREADS"] = "4"

import entity_forge  # noqa: F401  (caps thread pools; must come before polars)
import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(40); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(S.describe())

## Stage 1

In [ ]:
# Rows used to FIT the fold models come from the profile (S.stage1_max_rows); OOF scoring covers all pairs.
scores1 = stages.run_stage1_cv(S)
scores1

In [ ]:
imp = pl.concat([pl.read_csv(S.reports / f"importance_stage1_fold{f}.csv") for f in range(S.n_folds)])
imp.group_by("feature").agg(pl.col("gain").mean()).sort("gain", descending=True).head(25)

## Stage 2

In [ ]:
scores2 = stages.run_stage2_cv(S)
scores2

## Decision rule (exact metric, out-of-fold)

In [ ]:
best_s1, grid_s1 = stages.run_decision_tuning(S, score_col="p1")   # stage-1 only, for the ablation table
best, grid = stages.run_decision_tuning(S, score_col="p")           # final (written last -> frozen)
print("stage-1 best:", best_s1)
print("stage-2 best:", best)
grid.head(15)

## Per-country out-of-fold results

In [ ]:
stages.oof_report(S)

## Error analysis (OOF): wrong merges and missed matches

In [ ]:
false_pos, false_neg = stages.error_examples(S, n=15)
print("FALSE POSITIVES (wrong merges)"); display(false_pos)
print("FALSE NEGATIVES (missed; p = null means lost in blocking)"); display(false_neg)

## Calibration (OOF stage-2, streamed)

In [ ]:
from entity_forge import checkpoints as ck
files = [p for c in stages.countries(S, "train") for p in ck.data_files(S.scores("train", 2, c))]
(pl.scan_parquet(files).with_columns((pl.col("p") * 10).floor().clip(0, 9).alias("bin"))
   .group_by("bin").agg(pl.col("p").mean().alias("mean_p"), pl.col("label").mean().alias("hit_rate"), pl.len())
   .sort("bin").collect(engine="streaming"))

## Checkpoint status

In [ ]:
pl.DataFrame(stages.status_rows(S))